In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Config

In [18]:
import sys
from pathlib import Path

# Find the repo root by walking up until we hit src/A_Config.py -- see
# AA_Cam_pose_estimate_batches.ipynb for why this is more robust than Path.cwd().parent.
project_root = Path.cwd()
while not (project_root / "src" / "A_Config.py").exists():
    if project_root == project_root.parent:
        raise RuntimeError("couldn't find repo root (src/A_Config.py) above cwd")
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))
sys.path.insert(0, str(project_root /"src" / "Experiments" / "templates"))
print("project_root:", project_root)

from A_Config import set_case, case_dir
from B_video_processing import video_fps
from AA_cam_pose_sweeps import reset_sweep_state, run_density_sweep
case_name = "304_met_multi_4"
experiment = "explore"
set_case(case_name, experiment)  # must happen before any case_dir()/A_Config getter call below

source_dir   = case_dir() / "010_source"
videos = list(source_dir.glob("*.mp4"))
video_path = videos[0]
import cv2; total_frames = int(cv2.VideoCapture(str(video_path)).get(cv2.CAP_PROP_FRAME_COUNT))
print("total_frames",total_frames )
# Fixed video range -- only the frame DENSITY within this range varies across the sweep.
# Hardcoded per AI_coppertest3_vggt.ipynb's precedent (no reusable start/end-frame config exists yet).
start_3D_recon = 0
end_3D_recon = 600


fps = video_fps(video_path)
print(f"video fps: {fps:.3f}")

# Target frame counts to sweep, ascending -- ascending order matters for the
# OOM-precaution logic in run_density_sweep (more frames -> more GPU memory).
CAM_POSES = [25, 50, 75, 100, 125, 150, 175, 200, 225, 250,275,300]

# Fixed run order (user-specified) -- cheapest/most-reliable techniques first.
TECHNIQUE_ORDER = ["CUT3R","CUT3R_Revisit", "lingbot-map", "VGGT-Omega", "MegaSaM", "VGGT"]

# Which techniques are actually enabled -- same defaults as AA_Cam_pose_estimate_batches.ipynb.
FLAGS = {
    "CUT3R":       False,
    "CUT3R_Revisit": False,
    "lingbot-map": False,
    "VGGT-Omega":  True,
    "MegaSaM":     False,
    "VGGT":        False,
}

# What the sweep keeps on disk afterwards vs deletes -- see AA_cam_pose_sweeps.py's
# _cleanup_technique_output for exactly what each toggle controls.
DATA_STORAGE = {
    "keep_inputs":     False,  # extracted frames per density (020_frames_for_cam_poses)
    "keep_recon_data": False,  # full heavy technique output (images/depth/point-clouds/caches)
    "keep_cam_poses":  True,   # lightweight <technique_output>/CAM_poses/poses.csv per density
    "full_pose":       True,  # rotation too, not just translation, in the retained CSV
    "render_bev":      True,   # one <technique_output>/BEV/bev.png per technique/density --
                                # every technique gets a Reconstruction (natively for VGGT-Omega,
                                # via F_transpose_to_recon_objects.py adapters for the rest), then
                                # P_projection_mapping.BEV_tile_render_PM renders from it. Cheap
                                # to keep even when keep_recon_data=False.
}

project_root: /cs/student/project_msc/2025/cgvi/dbarker
total_frames 609
video fps: 25.000


## Reset sweep state

Deletes **only** this sweep's own output -- everything from running this sweep, nothing more, nothing less: every `poses_*` frame/reconstruction folder under the `experiment` tag defined above, plus the status/results CSVs. Nothing from `AA_Cam_pose_estimate_batches.ipynb` (different experiment name) or anything else in the case directory is touched.

Safe by default: with `CONFIRM_DELETE = False` this only prints what it *would* delete. Flip it to `True` and re-run the cell to actually delete.

In [19]:
CONFIRM_DELETE = False  # flip to True and re-run this cell to actually delete

reset_sweep_state(case_name, experiment, confirm=CONFIRM_DELETE)

confirm=False -- dry run only, nothing deleted. Would remove:
  (nothing to delete)


In [20]:
(end_3D_recon - start_3D_recon)

600

## Part 2 -- Sweep (expensive, resumable, run once/rarely)

`run_density_sweep` (in `AA_cam_pose_sweeps.py`) gives each `cam_poses` value its own frames/output folders via a nested experiment tag (`f"{experiment}/poses_{cam_poses:04d}"` passed to `set_case`) -- this makes every `A_Config` getter (including `run_lingbot_map`'s, which has no path parameters at all and only reads the zero-arg getters) automatically density-scoped, with no per-function path-wrangling needed.

Safe to interrupt/kill and re-run from the top at any point.

In [ ]:
run_density_sweep(
    case_name, experiment, video_path, start_3D_recon, end_3D_recon,
    CAM_POSES, TECHNIQUE_ORDER, FLAGS, DATA_STORAGE, project_root,
)

Video   : 04.mp4
          25 fps · 609 frames · 24.4s
Range   : frames 0-600 (600 frames)
Sampling: every 24 frames (0.96s) → 25 candidates
Window  : ±7 frames  |  scoring at 25% resolution


Pass 1: scoring: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 600/600 [00:01<00:00, 573.11fr/s]


Selected: 25 frames  (11 shifted from sample point, 0 skipped below threshold)


Pass 3: writing: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 600/600 [00:00<00:00, 829.49fr/s]


Log     : /cs/student/project_msc/2025/cgvi/dbarker/Data/304_met_multi_4/020_frames_for_cam_poses/explore/poses_0025/selection_log.json
Sharpness (at 25% res): min 633 · mean 1905 · max 3484
25 frames  →  /cs/student/project_msc/2025/cgvi/dbarker/Data/304_met_multi_4/035_VGGT_O_output/explore/poses_0025
Saved: /cs/student/project_msc/2025/cgvi/dbarker/Data/304_met_multi_4/035_VGGT_O_output/explore/poses_0025/predictions.npz
Saved: /cs/student/project_msc/2025/cgvi/dbarker/Data/304_met_multi_4/035_VGGT_O_output/explore/poses_0025/scene.glb
[0, 24, 50, 72, 93, 123, 144, 175, 197, 210, 240, 264, 288, 312, 336, 360, 384, 405, 432, 456, 479, 499, 526, 552, 580]
tensor(2.9765, device='cuda:0')
tensor(2.9689, device='cuda:0')
tensor(2.9765, device='cuda:0')
tensor(2.9922, device='cuda:0')
tensor(2.9531, device='cuda:0')
tensor(2.9689, device='cuda:0')
tensor(2.9531, device='cuda:0')
tensor(2.9922, device='cuda:0')
tensor(2.9843, device='cuda:0')
tensor(2.9844, device='cuda:0')
tensor(2.9922, 

...

Pass 1: scoring: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 600/600 [00:00<00:00, 736.10fr/s]


Selected: 300 frames  (25 shifted from sample point, 0 skipped below threshold)


Pass 3: writing: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 600/600 [00:01<00:00, 309.77fr/s]


Log     : /cs/student/project_msc/2025/cgvi/dbarker/Data/304_met_multi_4/020_frames_for_cam_poses/explore/poses_0300/selection_log.json
Sharpness (at 25% res): min 539 · mean 1756 · max 3438
[VGGT-Omega @ 300] already failed (skipped_precautionary), skipping


: 